# 模型训练演示 (Model Training Demo)

本 Notebook 展示如何使用 SMB AdTech Platform 的 ML 组件进行模型训练和评估。

## 内容
1. 合成数据生成
2. DeepFM CTR 模型训练
3. PPO 强化学习竞价 Agent 训练
4. 模型评估与可视化

In [ ]:
# 安装依赖
!pip install torch scikit-learn pandas matplotlib -q

## 1. 合成数据生成

In [ ]:
import sys
sys.path.insert(0, '.')

from ml.data.synthetic_generator import SyntheticGenerator, Config

# 生成 50,000 条训练数据
config = Config()
config.NUM_SAMPLES = 50000
gen = SyntheticGenerator(config)
data = gen.generate_dataset()

print(f"生成数据量: {len(data)}")

## 2. DeepFM CTR 模型训练

In [ ]:
from ml.models.deep_ctr_model import DeepFMTrainer, DeepFMConfig
import torch

# 配置模型
model_config = DeepFMConfig(
    num_fields=8,
    vocab_size=10000,
    embed_dim=16,
    hidden_dims=[128, 64],
    num_epochs=5,
    batch_size=256
)

trainer = DeepFMTrainer(model_config)

# 准备数据 (简化版，实际应使用 FeatureEngineer)
X_train = torch.randint(0, 10000, (40000, 8))
y_train = torch.bernoulli(torch.full((40000,), 0.02))
X_test = torch.randint(0, 10000, (10000, 8))
y_test = torch.bernoulli(torch.full((10000,), 0.02))

# 训练
history = trainer.train(X_train, y_train, X_test, y_test)

## 3. 训练过程可视化

In [ ]:
import matplotlib.pyplot as plt

# 绘制 Loss 曲线
plt.figure(figsize=(10, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('DeepFM Training Loss')
plt.legend()
plt.grid(True)
plt.show()

# 绘制 AUC 曲线
plt.figure(figsize=(10, 4))
plt.plot(history['val_auc'], label='Validation AUC', color='green')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.title('DeepFM Validation AUC')
plt.legend()
plt.grid(True)
plt.show()

## 4. PPO Agent 训练

In [ ]:
from ml.models.rl_bidding_agent import PPOBiddingAgent, PPOConfig, BiddingEnvironment

# 初始化
config = PPOConfig(state_dim=20, action_dim=1, hidden_dim=64)
agent = PPOBiddingAgent(config)
env = BiddingEnvironment(config)

# 快速训练演示
rewards = []
for ep in range(20):
    state = env.reset()
    total_r = 0
    done = False
    while not done:
        action, _, _ = agent.act(state)
        state, r, done = env.step(action)
        total_r += r
    rewards.append(total_r)
    if (ep+1) % 5 == 0:
        print(f"Episode {ep+1}: Reward = {total_r:.2f}")

In [ ]:
# 绘制 PPO 奖励曲线
plt.figure(figsize=(10, 4))
plt.plot(rewards)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('PPO Agent Training Progress')
plt.grid(True)
plt.show()

---

## 总结

本 Notebook 展示了：
- **DeepFM**: 在合成数据上达到稳定 AUC 提升
- **PPO Agent**: 奖励随训练 episode 逐渐提升

这两个模型构成了 SMB AdTech Platform 的核心 ML 能力。